Read Stream

In [0]:
df_stream = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.schemaLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/schema/") \
  .load("/Volumes/test/test_schema/my_volume/auto_loader/json/input/")

Write Stream - Location

In [0]:
df_stream.writeStream \
.option("checkpointLocation","/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint")\
    .trigger(availableNow = True)\
    .start("/Volumes/test/test_schema/my_volume/auto_loader/json/output")\
        .awaitTermination()

Write Stream - Delta

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.auto_loader_demo")

df_stream.writeStream \
.option("checkpointLocation","/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint2")\
    .trigger(availableNow = True)\
    .table("workspace.auto_loader_demo.demo_table")\
        .awaitTermination()

In [0]:
%sql
Select * from workspace.auto_loader_demo.demo_table;

customer_name,id,price,product,quantity,remarks,_rescued_data
John Doe,1,85000,"Laptop, 16GB RAM",2,Delivered on time,null
Jane Smith,2,55000,"Smartphone, 128GB",1,"Customer said ""excellent""",null
Bob Lee,3,1200,null,3,NULL,null
Alice Wong,4,2500,Keyboard,null,Missing quantity,null
null,5,null,"Monitor, 27-inch",2,No price info,null
Tom Brown,6,700,Mouse,5,"Delivered, no issues",null
Mark Davis,7,150,USB Cable,2,,null


In [0]:
%sql
select * from cloud_files_state('/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint5')

path,size,create_time,discovery_time,commit_time,archive_time,source_id,processed_time,ingestion_state,archive_mode,move_location
dbstorage-prod-x1rel/uc/f7a00d50-c41a-414c-a440-6c47f9b29010/6166d84c-a205-47d6-8ab6-b5f59f999a54/__unitystorage/catalogs/5b0ba22c-cfc2-46ab-88fb-4bdf3a26a587/volumes/f882163f-e319-4088-b302-2bb62755655c/auto_loader/json/input/sales_1.json,788,2026-10-23T06:00:37.000Z,2026-10-23T07:18:28.418Z,2026-10-23T07:21:41.553Z,null,0,2026-10-23T07:18:31.977Z,INGESTED,null,null
dbstorage-prod-x1rel/uc/f7a00d50-c41a-414c-a440-6c47f9b29010/6166d84c-a205-47d6-8ab6-b5f59f999a54/__unitystorage/catalogs/5b0ba22c-cfc2-46ab-88fb-4bdf3a26a587/volumes/f882163f-e319-4088-b302-2bb62755655c/auto_loader/json/input/sales_3.json,868,2026-10-23T07:23:05.000Z,2026-10-23T07:23:26.853Z,null,null,0,2026-10-23T07:23:30.567Z,INGESTED,null,null
dbstorage-prod-x1rel/uc/f7a00d50-c41a-414c-a440-6c47f9b29010/6166d84c-a205-47d6-8ab6-b5f59f999a54/__unitystorage/catalogs/5b0ba22c-cfc2-46ab-88fb-4bdf3a26a587/volumes/f882163f-e319-4088-b302-2bb62755655c/auto_loader/json/input/sales_4.json,860,2026-10-23T07:23:05.000Z,2026-10-23T07:23:26.855Z,null,null,0,2026-10-23T07:23:30.567Z,INGESTED,null,null
dbstorage-prod-x1rel/uc/f7a00d50-c41a-414c-a440-6c47f9b29010/6166d84c-a205-47d6-8ab6-b5f59f999a54/__unitystorage/catalogs/5b0ba22c-cfc2-46ab-88fb-4bdf3a26a587/volumes/f882163f-e319-4088-b302-2bb62755655c/auto_loader/json/input/sales_2.json,871,2026-10-23T07:21:19.000Z,2026-10-23T07:21:37.295Z,2026-10-23T07:23:31.103Z,null,0,2026-10-23T07:21:41.019Z,INGESTED,null,null


Schema Option 1 - Provide Schema

In [0]:
from pyspark.sql.types import StructType,StructField, IntegerType,StringType
schema = StructType([
  StructField("customer_name", StringType(), nullable=True),
  StructField("id", IntegerType(),nullable =  False),
  StructField("price", IntegerType(),nullable =  True),
  StructField("product", StringType(), nullable=True),
  StructField("quantity", IntegerType(),nullable =  True),
  StructField("remarks", StringType(), nullable=True)
  ])

spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .schema(schema)\
  .option("cloudFiles.schemaLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/schema3/") \
  .load("/Volumes/test/test_schema/my_volume/auto_loader/json/input/")\
    .writeStream \
  .option("checkpointLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint4") \
  .trigger(availableNow=True) \
  .table("workspace.auto_loader_demo.demo_table4")\
  .awaitTermination()

In [0]:
%sql
select * from workspace.auto_loader_demo.demo_table4 order by id desc

customer_name,id,price,product,quantity,remarks
Mark Davis,7,150,USB Cable,2,
Tom Brown,6,700,Mouse,5,"Delivered, no issues"
null,5,null,"Monitor, 27-inch",2,No price info
Alice Wong,4,2500,Keyboard,null,Missing quantity
Bob Lee,3,1200,null,3,NULL
Jane Smith,2,55000,"Smartphone, 128GB",1,"Customer said ""excellent"""
John Doe,1,85000,"Laptop, 16GB RAM",2,Delivered on time


Schema Option 2 - Infer Column Data Types

In [0]:
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.inferColumnTypes","true")\
  .option("cloudFiles.schemaLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/schema3/") \
  .load("/Volumes/test/test_schema/my_volume/auto_loader/json/input/")\
    .writeStream \
  .option("checkpointLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint5") \
  .trigger(availableNow=True) \
  .table("workspace.auto_loader_demo.demo_table5")\
  .awaitTermination()

In [0]:
%sql
select * from workspace.auto_loader_demo.demo_table5 order by id desc

customer_name,id,price,product,quantity,remarks,_rescued_data
Mark Davis,7,150,USB Cable,2,,null
Tom Brown,6,700,Mouse,5,"Delivered, no issues",null
null,5,null,"Monitor, 27-inch",2,No price info,null
Alice Wong,4,2500,Keyboard,null,Missing quantity,null
Bob Lee,3,1200,null,3,NULL,null
Jane Smith,2,55000,"Smartphone, 128GB",1,"Customer said ""excellent""",null
John Doe,1,85000,"Laptop, 16GB RAM",2,Delivered on time,null


Handle Schema Changes - Rescue Data

In [0]:
spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "json") \
  .option("cloudFiles.inferColumnTypes","true")\
  .option("cloudFiles.schemaEvolutionMode", "rescue")\
  .option("cloudFiles.schemaLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/schema3/") \
  .load("/Volumes/test/test_schema/my_volume/auto_loader/json/input/")\
    .writeStream \
  .option("checkpointLocation", "/Volumes/test/test_schema/my_volume/auto_loader/json/checkpoint5") \
  .trigger(availableNow=True) \
  .table("workspace.auto_loader_demo.demo_table5")\
  .awaitTermination()

In [0]:
%sql
select * from workspace.auto_loader_demo.demo_table5 order by id desc

customer_name,id,price,product,quantity,remarks,_rescued_data
Ayesha Khan,35,1500,null,5,Repeat business,"{""order_status"":""DELIVERED"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
Nikhil Bansal,34,6700,null,1,,"{""order_status"":""CANCELLED"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
null,33,null,null,3,Customer and price missing,"{""order_status"":""PENDING"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
James Wilson,32,2200,null,4,Bulk discount applied,"{""order_status"":""DELIVERED"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
Sunita Deshmukh,31,3400,null,null,Quantity not finalized,"{""order_status"":""ON_HOLD"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
Laura Martinez,30,12500,null,1,International order,"{""order_status"":""PROCESSING"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
Rakesh Gupta,29,4800,null,2,Priority delivery,"{""order_status"":""SHIPPED"",""_file_path"":""/Volumes/test/test_schema/my_volume/auto_loader/json/input/sales_5_schema_change.json""}"
Vikram Rao,28,2700,Gaming Mouse,3,Bulk order,null
null,27,null,"Portable SSD, 512GB",2,Price pending approval,null
Daniel Anderson,26,800,"HDMI Cable, 2m",null,Quantity to be verified,null
